# Notebook 03 — Model Training & Evaluation
**Project:** Flight–Bird Strike Risk Prediction and Mitigation System  
**Author:** A.D.P.I. Gunawardhana (CS/2020/011)  

Models trained:
1. Logistic Regression (interpretable baseline)
2. Random Forest
3. XGBoost
4. LightGBM

Both **multi-class** (4 damage levels) and **binary** (damaged vs not) tasks.

## 0. Environment Setup

In [ ]:
import sys, os

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    sys.path.insert(0, '/content/drive/MyDrive/Flight-Bird-Strike')
    import subprocess
    subprocess.run(['pip', 'install', 'xgboost', 'lightgbm', 'imbalanced-learn', '-q'])
else:
    sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report, accuracy_score, f1_score

from config import cfg
from src.data.loader import load_processed
from src.data.preprocessor import get_feature_columns, DAMAGE_LEVEL_LABELS
from src.models.train import train_and_evaluate, save_model, build_models, get_X_y
from src.models.evaluate import (
    plot_confusion_matrix, plot_feature_importance,
    print_classification_report, save_results_csv
)

sns.set_style('whitegrid')
plt.rcParams.update({'figure.dpi': 120})
print('Setup complete')

## 1. Load Preprocessed Data

In [ ]:
df = load_processed()
print(f"Shape: {df.shape}")
print(f"\nTarget distribution (damage_label):")
vc = df['damage_label'].value_counts().sort_index()
for k, v in vc.items():
    print(f"  {DAMAGE_LEVEL_LABELS.get(k, k)} ({k}): {v:,}  ({100*v/len(df):.1f}%)")

## 2. Train All Models — Multi-class Task
Predicting 4 damage levels: None / Minor / Substantial / Destroyed

In [ ]:
results_multi = train_and_evaluate(df, target='damage_label')

## 3. Train All Models — Binary Task
Predicting: No Damage (0) vs Damaged (1)

In [ ]:
results_binary = train_and_evaluate(df, target='damage_binary')

## 4. Model Comparison Chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, results, title in [
    (axes[0], results_multi, 'Multi-class (4 Damage Levels)'),
    (axes[1], results_binary, 'Binary (Damaged vs No Damage)')
]:
    x = np.arange(len(results))
    width = 0.35
    bars1 = ax.bar(x - width/2, results['Accuracy'], width, label='Accuracy',
                   color='steelblue', edgecolor='black')
    bars2 = ax.bar(x + width/2, results['F1_Weighted'], width, label='F1 (Weighted)',
                   color='coral', edgecolor='black')
    ax.set_xticks(x)
    ax.set_xticklabels(results['Model'], rotation=15, ha='right')
    ax.set_ylim(0, 1.05)
    ax.set_title(f'Model Comparison — {title}', fontweight='bold')
    ax.set_ylabel('Score')
    ax.legend()
    ax.axhline(y=0.9, color='green', linestyle='--', alpha=0.5, label='90% line')
    for bar in list(bars1) + list(bars2):
        ax.annotate(f'{bar.get_height():.3f}',
                    xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                    xytext=(0, 2), textcoords='offset points',
                    ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig(f'{cfg.FIGURES_DIR}/model_comparison.png', bbox_inches='tight')
plt.show()

## 5. Best Model — Detailed Evaluation

In [ ]:
# Pick best model from multi-class results
best_row = results_multi.iloc[0]
best_model = best_row['estimator']
best_name = best_row['Model']
X_test = best_row['X_test']
y_test = best_row['y_test']
feature_cols = best_row['feature_cols']

y_pred = best_model.predict(X_test)

print(f"Best Model: {best_name}")
print(f"Accuracy:   {accuracy_score(y_test, y_pred):.4f}")
print(f"F1 Score:   {f1_score(y_test, y_pred, average='weighted', zero_division=0):.4f}")
print()
print_classification_report(y_test, y_pred)

In [ ]:
# Confusion Matrix
plot_confusion_matrix(y_test, y_pred,
                      title=f'Confusion Matrix — {best_name}',
                      save=True)

In [ ]:
# Feature Importance
plot_feature_importance(best_model, feature_cols, top_n=20,
                        title=f'Feature Importance — {best_name}',
                        save=True)

## 6. Cross-Validation (5-fold)

In [ ]:
X_all, y_all, _ = get_X_y(df, target='damage_label')

cv = StratifiedKFold(n_splits=cfg.CV_FOLDS, shuffle=True, random_state=cfg.RANDOM_STATE)
cv_scores = cross_val_score(best_model, X_all, y_all, cv=cv,
                            scoring='f1_weighted', n_jobs=-1)

print(f"Cross-Validation Results ({cfg.CV_FOLDS}-fold) for {best_name}:")
print(f"  F1 per fold: {[round(s, 4) for s in cv_scores]}")
print(f"  Mean F1: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

## 7. Save Best Models

In [ ]:
# Save all trained models
model_paths = {
    'Random Forest': cfg.RF_MODEL_PATH,
    'XGBoost': cfg.XGB_MODEL_PATH,
    'LightGBM': cfg.LGB_MODEL_PATH,
}

for row in results_multi.to_dict('records'):
    name = row['Model']
    if name in model_paths:
        save_model(row['estimator'], model_paths[name])

# Save results table
save_results_csv(results_multi, 'model_comparison_multiclass.csv')
save_results_csv(results_binary, 'model_comparison_binary.csv')

print("\nAll models and results saved successfully!")

## 8. Results Summary

| Model | Accuracy | F1 (Weighted) | Notes |
|---|---|---|---|
| Logistic Regression | see above | see above | Fast, interpretable |
| Random Forest | see above | see above | Good baseline, feature importance |
| XGBoost | see above | see above | Usually best on tabular data |
| LightGBM | see above | see above | Fast + high accuracy |

**Key observations:**
- Tree-based models (XGBoost/LightGBM/RF) outperform logistic regression on this imbalanced dataset
- `class_weight='balanced'` helps models handle the dominant 'No Damage' class
- Feature importance reveals flight phase, altitude, and species size as top predictors

**Next steps (after mid-evaluation):**
- Hyperparameter tuning with GridSearchCV / Optuna
- SMOTE oversampling on minority classes
- Dashboard integration